In [1]:
# ===============================
# Wholesale Customer Segmentation
# K-Means + Agglomerative Clustering
# ===============================

import pandas as pd
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score

CSV_PATH = "dataset/raw_wholesale_customers.csv"
OUT_PATH = "dataset/segmented_wholesale_customers.csv"

FEATURES = [
    "Fresh",
    "Milk",
    "Grocery",
    "Frozen",
    "Detergents_Paper",
    "Delicassen"
]

K = 5
RANDOM_STATE = 42

In [2]:
# --------------------------------
# 1. Load Dataset
# --------------------------------
df = pd.read_csv(CSV_PATH)

print("\n=== DATASET HEAD ===")
print(df.head())

print("\n=== DATASET INFO ===")
df.info()


=== DATASET HEAD ===
   Channel  Region  Fresh  Milk  Grocery  Frozen  Detergents_Paper  Delicassen
0        2       3  12669  9656     7561     214              2674        1338
1        2       3   7057  9810     9568    1762              3293        1776
2        2       3   6353  8808     7684    2405              3516        7844
3        1       3  13265  1196     4221    6404               507        1788
4        2       3  22615  5410     7198    3915              1777        5185

=== DATASET INFO ===
<class 'pandas.DataFrame'>
RangeIndex: 440 entries, 0 to 439
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Channel           440 non-null    int64
 1   Region            440 non-null    int64
 2   Fresh             440 non-null    int64
 3   Milk              440 non-null    int64
 4   Grocery           440 non-null    int64
 5   Frozen            440 non-null    int64
 6   Detergents_Paper  440 no

In [3]:
# --------------------------------
# 2. Feature Selection + IQR Capping
# --------------------------------
X = df[FEATURES].copy()

def iqr_cap(series, k=1.5):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1

    lower = q1 - k * iqr
    upper = q3 + k * iqr

    return series.clip(lower=lower, upper=upper)

for col in FEATURES:
    X[col] = iqr_cap(X[col])

df[FEATURES] = X

print("\n=== AFTER IQR CAPPING ===")
print(X.head())


=== AFTER IQR CAPPING ===
     Fresh    Milk  Grocery  Frozen  Detergents_Paper  Delicassen
0  12669.0  9656.0   7561.0   214.0            2674.0     1338.00
1   7057.0  9810.0   9568.0  1762.0            3293.0     1776.00
2   6353.0  8808.0   7684.0  2405.0            3516.0     3938.25
3  13265.0  1196.0   4221.0  6404.0             507.0     1788.00
4  22615.0  5410.0   7198.0  3915.0            1777.0     3938.25


In [7]:
# 3. Standard Scaling
# --------------------------------
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print("\nScaling Complete")
print(X_scaled.shape)


Scaling Complete
(440, 6)


In [8]:
# 4. Elbow Method
# --------------------------------
print("\n=== ELBOW METHOD ===")

for k in range(1,11):

    km = KMeans(
        n_clusters=k,
        random_state=RANDOM_STATE,
        n_init="auto"
    )

    km.fit(X_scaled)

    print(f"k={k}  SSE={km.inertia_:.2f}")


=== ELBOW METHOD ===
k=1  SSE=2640.00
k=2  SSE=1728.19
k=3  SSE=1363.45
k=4  SSE=1202.41
k=5  SSE=1070.15
k=6  SSE=964.76
k=7  SSE=921.14
k=8  SSE=776.63
k=9  SSE=726.88
k=10  SSE=707.41


In [10]:
# --------------------------------
# 5. Train K-Means
# --------------------------------
kmeans = KMeans(
    n_clusters=K,
    random_state=RANDOM_STATE,
    n_init="auto"
)

kmeans_labels = kmeans.fit_predict(X_scaled)

df["Cluster"] = kmeans_labels

In [11]:
# --------------------------------
# 6. Evaluate K-Means
# --------------------------------
kmeans_sil = silhouette_score(
    X_scaled,
    kmeans_labels
)

kmeans_dbi = davies_bouldin_score(
    X_scaled,
    kmeans_labels
)

print("\n=== K-MEANS METRICS ===")
print(f"Silhouette Score : {kmeans_sil:.3f}")
print(f"Davies-Bouldin   : {kmeans_dbi:.3f}")


=== K-MEANS METRICS ===
Silhouette Score : 0.283
Davies-Bouldin   : 1.270


In [12]:
# --------------------------------
# 7. Cluster Centers
# --------------------------------
centers = scaler.inverse_transform(
    kmeans.cluster_centers_
)

centers = pd.DataFrame(
    centers,
    columns=FEATURES
)

print("\n=== K-MEANS CLUSTER CENTERS ===")
print(centers.round(2))



=== K-MEANS CLUSTER CENTERS ===
      Fresh      Milk   Grocery   Frozen  Detergents_Paper  Delicassen
0   9202.67   6833.30   9104.12  1326.16           3280.12     1871.76
1   8376.23   2150.65   3160.63  1646.33            779.25      674.02
2  17461.54  13805.60  17524.12  4120.57           5460.56     3583.64
3  22346.70   3409.14   3969.33  5819.60            583.07     1566.95
4   4916.98  10768.85  18350.13  1212.37           7780.02      981.37


In [13]:
# --------------------------------
# 8. Agglomerative Clustering
# --------------------------------
agg = AgglomerativeClustering(
    n_clusters=5
)

agg_labels = agg.fit_predict(X_scaled)

df["Agglomerative_Cluster"] = agg_labels

agg_sil = silhouette_score(
    X_scaled,
    agg_labels
)

print("\n=== AGGLOMERATIVE METRIC ===")
print(f"Silhouette Score : {agg_sil:.3f}")


=== AGGLOMERATIVE METRIC ===
Silhouette Score : 0.218


In [14]:
# --------------------------------
# 9. Compare Methods
# --------------------------------
print("\n=== COMPARISON ===")

print(f"K-Means Silhouette        : {kmeans_sil:.3f}")
print(f"Agglomerative Silhouette : {agg_sil:.3f}")

if kmeans_sil > agg_sil:
    print("K-Means produced better separated clusters.")
else:
    print("Agglomerative produced better separated clusters.")


=== COMPARISON ===
K-Means Silhouette        : 0.283
Agglomerative Silhouette : 0.218
K-Means produced better separated clusters.


In [15]:
# --------------------------------
# 10. Sanity Check
# --------------------------------
print("\n=== SANITY CHECK ===")

print(
    df.loc[
        [0,1,2],
        [
            "Channel",
            "Region",
            *FEATURES,
            "Cluster",
            "Agglomerative_Cluster"
        ]
    ]
)


=== SANITY CHECK ===
   Channel  Region    Fresh    Milk  Grocery  Frozen  Detergents_Paper  \
0        2       3  12669.0  9656.0   7561.0   214.0            2674.0   
1        2       3   7057.0  9810.0   9568.0  1762.0            3293.0   
2        2       3   6353.0  8808.0   7684.0  2405.0            3516.0   

   Delicassen  Cluster  Agglomerative_Cluster  
0     1338.00        0                      4  
1     1776.00        0                      4  
2     3938.25        0                      0  


In [16]:
# --------------------------------
# 11. Save CSV
# --------------------------------
df.to_csv(
    OUT_PATH,
    index=False
)

print(f"\nSaved file to: {OUT_PATH}")


Saved file to: dataset/segmented_wholesale_customers.csv
